# Transfer Learning Suite: Multi-Modal & Multi-Task

A comprehensive transfer learning pipeline supporting various modalities (image, text, tabular, multimodal) and tasks (classification, regression, segmentation, generation).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.models import ResNet50_Weights, EfficientNet_B0_Weights

# Advanced model libraries
try:
    import timm  # PyTorch Image Models
except ImportError:
    print("timm not installed. Run: pip install timm")
    
try:
    from transformers import AutoModel, AutoTokenizer, AutoConfig
except ImportError:
    print("transformers not installed. Run: pip install transformers")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
from typing import Dict, List, Optional, Union, Tuple, Any, Callable
from dataclasses import dataclass, field
from pathlib import Path
import json
import time
from collections import OrderedDict, defaultdict
from tqdm import tqdm
import pickle

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Universal Transfer Learning Framework

In [ ]:
@dataclass
class TransferConfig:
    """Configuration for transfer learning."""
    task_type: str = 'classification'  # 'classification', 'regression', 'segmentation', 'generation'
    modality: str = 'image'  # 'image', 'text', 'tabular', 'multimodal'
    backbone_name: str = 'resnet50'
    freeze_backbone: bool = True
    fine_tuning_strategy: str = 'gradual'  # 'none', 'full', 'gradual', 'discriminative'
    learning_rate: float = 1e-3
    backbone_lr_multiplier: float = 0.1
    num_epochs: int = 100
    batch_size: int = 32
    num_classes: Optional[int] = None
    hidden_dims: List[int] = field(default_factory=lambda: [512, 256])
    dropout_rate: float = 0.3
    use_mixup: bool = False
    mixup_alpha: float = 0.2
    label_smoothing: float = 0.1
    warmup_epochs: int = 5


class UniversalTransferLearning:
    """Universal transfer learning for multiple modalities."""
    
    def __init__(self, config: TransferConfig):
        self.config = config
        self.model = None
        self.backbone = None
        self.head = None
        self.preprocessor = None
        self.tokenizer = None
        self.feature_extractor = None
        self.history = defaultdict(list)
        
        # Build model
        self._build_model()
    
    def _build_model(self) -> None:
        """Build complete model with backbone and head."""
        # Load pretrained backbone
        self.backbone = self._load_backbone()
        
        # Get feature dimensions
        feature_dim = self._get_feature_dim()
        
        # Create task-specific head
        self.head = self._create_task_head(feature_dim)
        
        # Combine into single model
        self.model = nn.Sequential(
            OrderedDict([
                ('backbone', self.backbone),
                ('head', self.head)
            ])
        ).to(device)
    
    def _load_backbone(self) -> nn.Module:
        """Load pretrained backbone for different modalities."""
        if self.config.modality == 'image':
            return self._load_image_backbone()
        elif self.config.modality == 'text':
            return self._load_text_backbone()
        elif self.config.modality == 'tabular':
            return self._load_tabular_backbone()
        elif self.config.modality == 'multimodal':
            return self._load_multimodal_backbone()
        else:
            raise ValueError(f"Unknown modality: {self.config.modality}")
    
    def _load_image_backbone(self) -> nn.Module:
        """Load image backbone with various architectures."""
        model_name = self.config.backbone_name
        
        if model_name.startswith('timm/'):
            # Use timm library for cutting-edge architectures
            import timm
            model = timm.create_model(
                model_name.replace('timm/', ''),
                pretrained=True,
                features_only=False,
                num_classes=0  # Remove classification head
            )
        elif model_name == 'resnet50':
            model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            # Remove final classification layer
            model = nn.Sequential(*list(model.children())[:-1])
            # Add adaptive pooling to handle different input sizes
            model.add_module('adaptive_pool', nn.AdaptiveAvgPool2d((1, 1)))
            model.add_module('flatten', nn.Flatten())
        elif model_name == 'resnet101':
            model = models.resnet101(pretrained=True)
            model = nn.Sequential(*list(model.children())[:-1])
            model.add_module('adaptive_pool', nn.AdaptiveAvgPool2d((1, 1)))
            model.add_module('flatten', nn.Flatten())
        elif model_name == 'efficientnet_b0':
            model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
            # Remove classifier
            model.classifier = nn.Identity()
        elif model_name == 'vit_base':
            import timm
            model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        elif model_name == 'densenet121':
            model = models.densenet121(pretrained=True)
            model = nn.Sequential(*list(model.children())[:-1])
            model.add_module('adaptive_pool', nn.AdaptiveAvgPool2d((1, 1)))
            model.add_module('flatten', nn.Flatten())
        else:
            raise ValueError(f"Unknown image model: {model_name}")
        
        # Freeze backbone if specified
        if self.config.freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        
        # Setup data preprocessing
        self.preprocessor = self._get_image_preprocessor(model_name)
        
        return model
    
    def _load_text_backbone(self) -> nn.Module:
        """Load text backbone from HuggingFace."""
        from transformers import AutoModel, AutoTokenizer
        
        model_name = self.config.backbone_name
        
        # Load model and tokenizer
        model = AutoModel.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Freeze backbone if specified
        if self.config.freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        
        # Wrap in a module that extracts features
        class TextFeatureExtractor(nn.Module):
            def __init__(self, transformer):
                super().__init__()
                self.transformer = transformer
            
            def forward(self, input_ids, attention_mask):
                outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
                # Use [CLS] token representation or pooled output
                return outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs.last_hidden_state[:, 0]
        
        return TextFeatureExtractor(model)
    
    def _load_tabular_backbone(self) -> nn.Module:
        """Load tabular backbone (AutoGluon TabNet or custom)."""
        
        class TabularBackbone(nn.Module):
            """Custom tabular backbone with entity embeddings."""
            
            def __init__(self, input_dim: int, hidden_dims: List[int] = [256, 128, 64]):
                super().__init__()
                
                layers = []
                prev_dim = input_dim
                
                for hidden_dim in hidden_dims:
                    layers.extend([
                        nn.Linear(prev_dim, hidden_dim),
                        nn.BatchNorm1d(hidden_dim),
                        nn.ReLU(),
                        nn.Dropout(0.2)
                    ])
                    prev_dim = hidden_dim
                
                self.feature_extractor = nn.Sequential(*layers)
                self.output_dim = hidden_dims[-1]
            
            def forward(self, x):
                return self.feature_extractor(x)
        
        # For demo, use a simple MLP backbone
        # In practice, you might load a pretrained TabNet or other model
        return TabularBackbone(100)  # Assuming 100 input features
    
    def _load_multimodal_backbone(self) -> nn.Module:
        """Load multimodal backbone (e.g., CLIP, ALIGN)."""
        
        class MultiModalBackbone(nn.Module):
            """Multimodal backbone combining image and text."""
            
            def __init__(self):
                super().__init__()
                
                # Image branch
                self.image_backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
                self.image_backbone = nn.Sequential(*list(self.image_backbone.children())[:-1])
                
                # Text branch (simplified)
                self.text_embedding = nn.Embedding(10000, 256)  # Vocab size 10000
                self.text_lstm = nn.LSTM(256, 128, batch_first=True)
                
                # Fusion layer
                self.fusion = nn.Sequential(
                    nn.Linear(2048 + 128, 512),  # ResNet50 output + LSTM output
                    nn.ReLU(),
                    nn.Dropout(0.3)
                )
            
            def forward(self, image, text):
                # Process image
                image_features = self.image_backbone(image)
                image_features = image_features.view(image_features.size(0), -1)
                
                # Process text
                text_embed = self.text_embedding(text)
                _, (text_features, _) = self.text_lstm(text_embed)
                text_features = text_features.squeeze(0)
                
                # Fuse features
                combined = torch.cat([image_features, text_features], dim=1)
                fused = self.fusion(combined)
                
                return fused
        
        model = MultiModalBackbone()
        
        if self.config.freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        
        return model
    
    def _get_feature_dim(self) -> int:
        """Get output dimension of backbone."""
        if self.config.modality == 'image':
            if 'resnet50' in self.config.backbone_name:
                return 2048
            elif 'resnet101' in self.config.backbone_name:
                return 2048
            elif 'efficientnet_b0' in self.config.backbone_name:
                return 1280
            elif 'vit' in self.config.backbone_name:
                return 768
            elif 'densenet121' in self.config.backbone_name:
                return 1024
            else:
                return 512  # Default
        elif self.config.modality == 'text':
            if 'bert-base' in self.config.backbone_name:
                return 768
            elif 'bert-large' in self.config.backbone_name:
                return 1024
            else:
                return 768  # Default
        elif self.config.modality == 'tabular':
            return 64  # From TabularBackbone
        elif self.config.modality == 'multimodal':
            return 512  # From MultiModalBackbone fusion layer
        else:
            return 512  # Default
    
    def _create_task_head(self, input_dim: int) -> nn.Module:
        """Create task-specific head."""
        
        layers = []
        prev_dim = input_dim
        
        # Hidden layers
        for hidden_dim in self.config.hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(self.config.dropout_rate)
            ])
            prev_dim = hidden_dim
        
        # Output layer based on task
        if self.config.task_type == 'classification':
            layers.append(nn.Linear(prev_dim, self.config.num_classes))
        elif self.config.task_type == 'regression':
            layers.append(nn.Linear(prev_dim, 1))
        elif self.config.task_type == 'multilabel':
            layers.append(nn.Linear(prev_dim, self.config.num_classes))
            layers.append(nn.Sigmoid())
        elif self.config.task_type == 'segmentation':
            # For segmentation, we need a decoder
            return self._create_segmentation_decoder(prev_dim)
        
        return nn.Sequential(*layers)
    
    def _create_segmentation_decoder(self, input_dim: int) -> nn.Module:
        """Create decoder for segmentation tasks."""
        
        class SegmentationDecoder(nn.Module):
            def __init__(self, input_dim, num_classes):
                super().__init__()
                
                self.decoder = nn.Sequential(
                    nn.ConvTranspose2d(input_dim, 256, 4, 2, 1),
                    nn.BatchNorm2d(256),
                    nn.ReLU(),
                    nn.ConvTranspose2d(256, 128, 4, 2, 1),
                    nn.BatchNorm2d(128),
                    nn.ReLU(),
                    nn.ConvTranspose2d(128, 64, 4, 2, 1),
                    nn.BatchNorm2d(64),
                    nn.ReLU(),
                    nn.ConvTranspose2d(64, num_classes, 4, 2, 1)
                )
            
            def forward(self, x):
                # Reshape if needed
                if len(x.shape) == 2:
                    x = x.view(x.size(0), -1, 1, 1)
                return self.decoder(x)
        
        return SegmentationDecoder(input_dim, self.config.num_classes)
    
    def _get_image_preprocessor(self, model_name: str) -> transforms.Compose:
        """Get appropriate preprocessor for image model."""
        
        if 'efficientnet' in model_name:
            return transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
            ])
        elif 'vit' in model_name:
            return transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5],
                                   std=[0.5, 0.5, 0.5])
            ])
        else:
            # Default ImageNet preprocessing
            return transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
            ])
    
    def freeze_backbone(self) -> None:
        """Freeze backbone parameters."""
        for param in self.backbone.parameters():
            param.requires_grad = False
    
    def unfreeze_backbone(self, unfreeze_last_n: Optional[int] = None) -> None:
        """Unfreeze backbone parameters."""
        params = list(self.backbone.parameters())
        
        if unfreeze_last_n is None:
            # Unfreeze all
            for param in params:
                param.requires_grad = True
        else:
            # Unfreeze last n layers
            for param in params[-unfreeze_last_n:]:
                param.requires_grad = True

## 2. Advanced Fine-Tuning Strategies

In [ ]:
class FineTuningStrategies:
    """Advanced fine-tuning strategies for transfer learning."""
    
    @staticmethod
    def gradual_unfreezing(model: UniversalTransferLearning,
                          train_loader: DataLoader,
                          val_loader: DataLoader,
                          optimizer: torch.optim.Optimizer,
                          criterion: nn.Module,
                          epochs_per_unfreeze: int = 5,
                          unfreeze_layers_per_step: int = 2) -> Dict[str, List]:
        """Gradually unfreeze layers during training."""
        
        history = defaultdict(list)
        backbone_layers = list(model.backbone.children())
        total_layers = len(backbone_layers)
        
        # Start with frozen backbone
        model.freeze_backbone()
        
        # Train head first
        print("Phase 1: Training head only...")
        for epoch in range(epochs_per_unfreeze):
            train_loss, train_acc = FineTuningStrategies._train_epoch(
                model.model, train_loader, optimizer, criterion
            )
            val_loss, val_acc = FineTuningStrategies._validate(
                model.model, val_loader, criterion
            )
            
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            
            print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        # Gradually unfreeze layers
        unfrozen_layers = 0
        phase = 2
        
        while unfrozen_layers < total_layers:
            # Unfreeze next batch of layers
            layers_to_unfreeze = min(unfreeze_layers_per_step, total_layers - unfrozen_layers)
            
            print(f"\nPhase {phase}: Unfreezing {layers_to_unfreeze} layers...")
            
            # Unfreeze from the end (closest to output)
            for i in range(layers_to_unfreeze):
                layer_idx = total_layers - unfrozen_layers - i - 1
                if layer_idx >= 0:
                    for param in backbone_layers[layer_idx].parameters():
                        param.requires_grad = True
            
            unfrozen_layers += layers_to_unfreeze
            
            # Train with newly unfrozen layers
            for epoch in range(epochs_per_unfreeze):
                train_loss, train_acc = FineTuningStrategies._train_epoch(
                    model.model, train_loader, optimizer, criterion
                )
                val_loss, val_acc = FineTuningStrategies._validate(
                    model.model, val_loader, criterion
                )
                
                history['train_loss'].append(train_loss)
                history['train_acc'].append(train_acc)
                history['val_loss'].append(val_loss)
                history['val_acc'].append(val_acc)
                history['phase'].append(phase)
                
                print(f"Epoch {len(history['train_loss'])}: Train Loss: {train_loss:.4f}, Val Acc: {val_acc:.2f}%")
            
            phase += 1
        
        return dict(history)
    
    @staticmethod
    def discriminative_learning_rates(model: UniversalTransferLearning,
                                     base_lr: float = 1e-3,
                                     lr_decay_factor: float = 0.3) -> torch.optim.Optimizer:
        """Apply different learning rates to different layers."""
        
        param_groups = []
        
        # Backbone layers with decreasing learning rates
        backbone_layers = list(model.backbone.children())
        n_layers = len(backbone_layers)
        
        for i, layer in enumerate(backbone_layers):
            layer_lr = base_lr * (lr_decay_factor ** (n_layers - i - 1))
            param_groups.append({
                'params': layer.parameters(),
                'lr': layer_lr
            })
        
        # Head with base learning rate
        param_groups.append({
            'params': model.head.parameters(),
            'lr': base_lr
        })
        
        return torch.optim.AdamW(param_groups)
    
    @staticmethod
    def cyclical_unfreezing(model: UniversalTransferLearning,
                          train_loader: DataLoader,
                          val_loader: DataLoader,
                          optimizer: torch.optim.Optimizer,
                          criterion: nn.Module,
                          cycles: int = 3,
                          epochs_per_cycle: int = 10) -> Dict[str, List]:
        """Cyclically freeze and unfreeze layers."""
        
        history = defaultdict(list)
        
        for cycle in range(cycles):
            print(f"\nCycle {cycle + 1}/{cycles}")
            
            # Freeze phase
            print("Freezing backbone...")
            model.freeze_backbone()
            
            for epoch in range(epochs_per_cycle // 2):
                train_loss, train_acc = FineTuningStrategies._train_epoch(
                    model.model, train_loader, optimizer, criterion
                )
                val_loss, val_acc = FineTuningStrategies._validate(
                    model.model, val_loader, criterion
                )
                
                history['train_loss'].append(train_loss)
                history['train_acc'].append(train_acc)
                history['val_loss'].append(val_loss)
                history['val_acc'].append(val_acc)
                history['frozen'].append(True)
            
            # Unfreeze phase
            print("Unfreezing backbone...")
            model.unfreeze_backbone()
            
            # Reduce learning rate for unfrozen phase
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 0.5
            
            for epoch in range(epochs_per_cycle // 2):
                train_loss, train_acc = FineTuningStrategies._train_epoch(
                    model.model, train_loader, optimizer, criterion
                )
                val_loss, val_acc = FineTuningStrategies._validate(
                    model.model, val_loader, criterion
                )
                
                history['train_loss'].append(train_loss)
                history['train_acc'].append(train_acc)
                history['val_loss'].append(val_loss)
                history['val_acc'].append(val_acc)
                history['frozen'].append(False)
            
            # Restore learning rate
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 2
        
        return dict(history)
    
    @staticmethod
    def _train_epoch(model: nn.Module,
                    dataloader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: nn.Module) -> Tuple[float, float]:
        """Train for one epoch."""
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            
            if isinstance(outputs, dict):
                outputs = outputs['logits']
            
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if len(outputs.shape) > 1 and outputs.shape[1] > 1:  # Classification
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
        
        avg_loss = running_loss / len(dataloader)
        accuracy = 100. * correct / total if total > 0 else 0
        
        return avg_loss, accuracy
    
    @staticmethod
    def _validate(model: nn.Module,
                 dataloader: DataLoader,
                 criterion: nn.Module) -> Tuple[float, float]:
        """Validate model."""
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, targets in dataloader:
                inputs, targets = inputs.to(device), targets.to(device)
                
                outputs = model(inputs)
                
                if isinstance(outputs, dict):
                    outputs = outputs['logits']
                
                loss = criterion(outputs, targets)
                running_loss += loss.item()
                
                if len(outputs.shape) > 1 and outputs.shape[1] > 1:  # Classification
                    _, predicted = outputs.max(1)
                    total += targets.size(0)
                    correct += predicted.eq(targets).sum().item()
        
        avg_loss = running_loss / len(dataloader)
        accuracy = 100. * correct / total if total > 0 else 0
        
        return avg_loss, accuracy

## 3. Domain Adaptation Techniques

In [ ]:
class DomainAdaptation:
    """Domain adaptation techniques for transfer learning."""
    
    @staticmethod
    def adversarial_domain_adaptation(feature_extractor: nn.Module,
                                     task_classifier: nn.Module,
                                     domain_discriminator: nn.Module,
                                     source_loader: DataLoader,
                                     target_loader: DataLoader,
                                     num_epochs: int = 50,
                                     lambda_adv: float = 1.0) -> Dict[str, List]:
        """Domain Adversarial Neural Networks (DANN) implementation."""
        
        # Optimizers
        feature_opt = torch.optim.Adam(feature_extractor.parameters(), lr=1e-4)
        task_opt = torch.optim.Adam(task_classifier.parameters(), lr=1e-3)
        domain_opt = torch.optim.Adam(domain_discriminator.parameters(), lr=1e-3)
        
        # Losses
        task_criterion = nn.CrossEntropyLoss()
        domain_criterion = nn.BCEWithLogitsLoss()
        
        history = defaultdict(list)
        
        for epoch in range(num_epochs):
            feature_extractor.train()
            task_classifier.train()
            domain_discriminator.train()
            
            source_iter = iter(source_loader)
            target_iter = iter(target_loader)
            
            epoch_task_loss = 0
            epoch_domain_loss = 0
            
            for _ in range(min(len(source_loader), len(target_loader))):
                # Get source and target batches
                try:
                    source_data, source_labels = next(source_iter)
                except StopIteration:
                    source_iter = iter(source_loader)
                    source_data, source_labels = next(source_iter)
                
                try:
                    target_data, _ = next(target_iter)
                except StopIteration:
                    target_iter = iter(target_loader)
                    target_data, _ = next(target_iter)
                
                source_data = source_data.to(device)
                source_labels = source_labels.to(device)
                target_data = target_data.to(device)
                
                # Create domain labels
                source_domain = torch.ones(source_data.size(0), 1).to(device)
                target_domain = torch.zeros(target_data.size(0), 1).to(device)
                
                # === Train domain discriminator ===
                domain_opt.zero_grad()
                
                # Source domain
                source_features = feature_extractor(source_data).detach()
                source_domain_pred = domain_discriminator(source_features)
                source_domain_loss = domain_criterion(source_domain_pred, source_domain)
                
                # Target domain
                target_features = feature_extractor(target_data).detach()
                target_domain_pred = domain_discriminator(target_features)
                target_domain_loss = domain_criterion(target_domain_pred, target_domain)
                
                domain_loss = source_domain_loss + target_domain_loss
                domain_loss.backward()
                domain_opt.step()
                
                # === Train feature extractor and task classifier ===
                feature_opt.zero_grad()
                task_opt.zero_grad()
                
                # Task loss on source domain
                source_features = feature_extractor(source_data)
                source_task_pred = task_classifier(source_features)
                task_loss = task_criterion(source_task_pred, source_labels)
                
                # Adversarial loss - fool the domain discriminator
                source_domain_pred = domain_discriminator(source_features)
                target_features = feature_extractor(target_data)
                target_domain_pred = domain_discriminator(target_features)
                
                # Reverse the domain labels to fool discriminator
                adv_loss = domain_criterion(source_domain_pred, target_domain) + \
                          domain_criterion(target_domain_pred, source_domain)
                
                total_loss = task_loss - lambda_adv * adv_loss
                total_loss.backward()
                
                feature_opt.step()
                task_opt.step()
                
                epoch_task_loss += task_loss.item()
                epoch_domain_loss += domain_loss.item()
            
            avg_task_loss = epoch_task_loss / min(len(source_loader), len(target_loader))
            avg_domain_loss = epoch_domain_loss / min(len(source_loader), len(target_loader))
            
            history['task_loss'].append(avg_task_loss)
            history['domain_loss'].append(avg_domain_loss)
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Task Loss: {avg_task_loss:.4f}, Domain Loss: {avg_domain_loss:.4f}")
        
        return dict(history)
    
    @staticmethod
    def maximum_mean_discrepancy(source_features: torch.Tensor,
                                target_features: torch.Tensor,
                                kernel: str = 'rbf',
                                gamma: float = 1.0) -> torch.Tensor:
        """Compute Maximum Mean Discrepancy (MMD) between source and target."""
        
        def rbf_kernel(X, Y, gamma):
            """RBF kernel for MMD."""
            XX = torch.mm(X, X.t())
            XY = torch.mm(X, Y.t())
            YY = torch.mm(Y, Y.t())
            
            X_sqr = torch.diag(XX).unsqueeze(1)
            Y_sqr = torch.diag(YY).unsqueeze(0)
            
            K_XX = torch.exp(-gamma * (X_sqr - 2 * XX + X_sqr.t()))
            K_XY = torch.exp(-gamma * (X_sqr - 2 * XY + Y_sqr))
            K_YY = torch.exp(-gamma * (Y_sqr.t() - 2 * YY + Y_sqr))
            
            return K_XX, K_XY, K_YY
        
        n_source = source_features.size(0)
        n_target = target_features.size(0)
        
        if kernel == 'rbf':
            K_XX, K_XY, K_YY = rbf_kernel(source_features, target_features, gamma)
        else:
            # Linear kernel
            K_XX = torch.mm(source_features, source_features.t())
            K_XY = torch.mm(source_features, target_features.t())
            K_YY = torch.mm(target_features, target_features.t())
        
        mmd = K_XX.mean() + K_YY.mean() - 2 * K_XY.mean()
        
        return mmd
    
    @staticmethod
    def coral_loss(source_features: torch.Tensor,
                  target_features: torch.Tensor) -> torch.Tensor:
        """Correlation Alignment (CORAL) loss."""
        
        d = source_features.size(1)
        
        # Source covariance
        source_mean = source_features.mean(0, keepdim=True)
        source_centered = source_features - source_mean
        source_cov = torch.mm(source_centered.t(), source_centered) / (source_features.size(0) - 1)
        
        # Target covariance
        target_mean = target_features.mean(0, keepdim=True)
        target_centered = target_features - target_mean
        target_cov = torch.mm(target_centered.t(), target_centered) / (target_features.size(0) - 1)
        
        # Frobenius norm of the difference
        loss = torch.norm(source_cov - target_cov, p='fro') / (4 * d * d)
        
        return loss

## 4. Few-Shot and Zero-Shot Learning

In [ ]:
class FewShotLearning:
    """Few-shot and zero-shot learning techniques."""
    
    @staticmethod
    def prototypical_networks(support_set: torch.Tensor,
                             support_labels: torch.Tensor,
                             query_set: torch.Tensor,
                             feature_extractor: nn.Module) -> torch.Tensor:
        """Prototypical Networks for few-shot learning."""
        
        feature_extractor.eval()
        
        with torch.no_grad():
            # Extract features
            support_features = feature_extractor(support_set)
            query_features = feature_extractor(query_set)
            
            # Compute prototypes (class centers)
            unique_labels = torch.unique(support_labels)
            prototypes = []
            
            for label in unique_labels:
                mask = support_labels == label
                class_features = support_features[mask]
                prototype = class_features.mean(0)
                prototypes.append(prototype)
            
            prototypes = torch.stack(prototypes)
            
            # Compute distances from query to prototypes
            distances = torch.cdist(query_features, prototypes)
            
            # Predict using nearest prototype
            predictions = distances.argmin(dim=1)
            
            # Map back to original labels
            predicted_labels = unique_labels[predictions]
        
        return predicted_labels
    
    @staticmethod
    def matching_networks(support_set: torch.Tensor,
                        support_labels: torch.Tensor,
                        query_set: torch.Tensor,
                        feature_extractor: nn.Module,
                        use_fce: bool = True) -> torch.Tensor:
        """Matching Networks for few-shot learning."""
        
        feature_extractor.eval()
        
        with torch.no_grad():
            # Extract features
            support_features = feature_extractor(support_set)
            query_features = feature_extractor(query_set)
            
            if use_fce:
                # Full Context Embeddings (simplified)
                # In practice, this would use LSTM or attention
                context = support_features.mean(0, keepdim=True)
                support_features = support_features + 0.1 * context
                query_features = query_features + 0.1 * context
            
            # Compute attention weights
            attention = F.softmax(torch.mm(query_features, support_features.t()), dim=1)
            
            # One-hot encode support labels
            unique_labels = torch.unique(support_labels)
            n_classes = len(unique_labels)
            support_one_hot = F.one_hot(support_labels, n_classes).float()
            
            # Weighted sum of support labels
            predictions = torch.mm(attention, support_one_hot)
            predicted_labels = predictions.argmax(dim=1)
        
        return predicted_labels
    
    @staticmethod
    def maml_inner_loop(model: nn.Module,
                       support_data: torch.Tensor,
                       support_labels: torch.Tensor,
                       inner_lr: float = 0.01,
                       inner_steps: int = 5) -> nn.Module:
        """Model-Agnostic Meta-Learning (MAML) inner loop."""
        
        # Clone model for inner loop
        inner_model = type(model)()  # Create new instance
        inner_model.load_state_dict(model.state_dict())
        inner_model.to(device)
        
        criterion = nn.CrossEntropyLoss()
        
        # Inner loop optimization
        for _ in range(inner_steps):
            outputs = inner_model(support_data)
            loss = criterion(outputs, support_labels)
            
            # Manual gradient descent
            grads = torch.autograd.grad(loss, inner_model.parameters(), create_graph=True)
            
            for param, grad in zip(inner_model.parameters(), grads):
                param.data = param.data - inner_lr * grad
        
        return inner_model

## 5. Model Fusion and Ensemble

In [ ]:
class ModelFusion:
    """Model fusion and ensemble techniques for transfer learning."""
    
    def __init__(self, models: List[nn.Module]):
        self.models = models
        self.fusion_weights = None
    
    def weighted_average_fusion(self, inputs: torch.Tensor,
                               weights: Optional[List[float]] = None) -> torch.Tensor:
        """Weighted average fusion of multiple models."""
        
        if weights is None:
            weights = [1.0 / len(self.models)] * len(self.models)
        
        outputs = []
        for model, weight in zip(self.models, weights):
            model.eval()
            with torch.no_grad():
                output = model(inputs)
                if isinstance(output, dict):
                    output = output['logits']
                outputs.append(weight * F.softmax(output, dim=1))
        
        return torch.stack(outputs).sum(0)
    
    def stacking_fusion(self, train_features: List[torch.Tensor],
                       train_labels: torch.Tensor,
                       val_features: List[torch.Tensor]) -> torch.Tensor:
        """Stacking ensemble with meta-learner."""
        
        # Concatenate base model predictions
        train_meta_features = torch.cat(train_features, dim=1)
        val_meta_features = torch.cat(val_features, dim=1)
        
        # Train meta-learner
        input_dim = train_meta_features.size(1)
        num_classes = train_labels.max().item() + 1
        
        meta_learner = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        ).to(device)
        
        optimizer = torch.optim.Adam(meta_learner.parameters())
        criterion = nn.CrossEntropyLoss()
        
        # Train meta-learner
        for epoch in range(50):
            optimizer.zero_grad()
            outputs = meta_learner(train_meta_features)
            loss = criterion(outputs, train_labels)
            loss.backward()
            optimizer.step()
        
        # Predict on validation
        meta_learner.eval()
        with torch.no_grad():
            predictions = meta_learner(val_meta_features)
        
        return predictions
    
    def knowledge_distillation_fusion(self, student_model: nn.Module,
                                    train_loader: DataLoader,
                                    temperature: float = 3.0,
                                    alpha: float = 0.7,
                                    epochs: int = 50) -> nn.Module:
        """Fuse multiple models through knowledge distillation."""
        
        optimizer = torch.optim.Adam(student_model.parameters())
        criterion_hard = nn.CrossEntropyLoss()
        criterion_soft = nn.KLDivLoss(reduction='batchmean')
        
        student_model.train()
        
        for epoch in range(epochs):
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # Get teacher predictions (ensemble)
                teacher_outputs = []
                for teacher in self.models:
                    teacher.eval()
                    with torch.no_grad():
                        output = teacher(inputs)
                        if isinstance(output, dict):
                            output = output['logits']
                        teacher_outputs.append(F.softmax(output / temperature, dim=1))
                
                # Average teacher predictions
                teacher_probs = torch.stack(teacher_outputs).mean(0)
                
                # Student predictions
                student_outputs = student_model(inputs)
                if isinstance(student_outputs, dict):
                    student_outputs = student_outputs['logits']
                
                # Hard loss (with true labels)
                hard_loss = criterion_hard(student_outputs, labels)
                
                # Soft loss (with teacher predictions)
                student_probs = F.log_softmax(student_outputs / temperature, dim=1)
                soft_loss = criterion_soft(student_probs, teacher_probs) * (temperature ** 2)
                
                # Combined loss
                loss = alpha * soft_loss + (1 - alpha) * hard_loss
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Loss: {loss.item():.4f}")
        
        return student_model

## 6. Practical Examples

In [ ]:
# Example 1: Image Classification Transfer Learning
def image_classification_example():
    """Example: Transfer learning for image classification."""
    
    print("=" * 50)
    print("Image Classification Transfer Learning Example")
    print("=" * 50)
    
    # Configuration
    config = TransferConfig(
        task_type='classification',
        modality='image',
        backbone_name='resnet50',
        num_classes=10,
        freeze_backbone=True,
        fine_tuning_strategy='gradual'
    )
    
    # Create model
    model = UniversalTransferLearning(config)
    print(f"Model created with {config.backbone_name} backbone")
    
    # Create dummy data
    train_data = torch.randn(100, 3, 224, 224)
    train_labels = torch.randint(0, 10, (100,))
    val_data = torch.randn(20, 3, 224, 224)
    val_labels = torch.randint(0, 10, (20,))
    
    train_dataset = TensorDataset(train_data, train_labels)
    val_dataset = TensorDataset(val_data, val_labels)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    
    # Setup training
    optimizer = FineTuningStrategies.discriminative_learning_rates(model, base_lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    # Train with gradual unfreezing
    history = FineTuningStrategies.gradual_unfreezing(
        model, train_loader, val_loader, optimizer, criterion,
        epochs_per_unfreeze=2, unfreeze_layers_per_step=2
    )
    
    print("\nTraining completed!")
    print(f"Final validation accuracy: {history['val_acc'][-1]:.2f}%")
    
    return model, history

# Example 2: Few-Shot Learning
def few_shot_learning_example():
    """Example: Few-shot learning with prototypical networks."""
    
    print("\n" + "=" * 50)
    print("Few-Shot Learning Example")
    print("=" * 50)
    
    # Create a simple feature extractor
    feature_extractor = nn.Sequential(
        nn.Conv2d(3, 64, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten()
    ).to(device)
    
    # Create few-shot task (5-way, 5-shot)
    n_way = 5  # Number of classes
    k_shot = 5  # Number of examples per class
    
    # Support set
    support_set = torch.randn(n_way * k_shot, 3, 32, 32).to(device)
    support_labels = torch.repeat_interleave(torch.arange(n_way), k_shot).to(device)
    
    # Query set
    query_set = torch.randn(n_way * 3, 3, 32, 32).to(device)  # 3 query per class
    query_labels = torch.repeat_interleave(torch.arange(n_way), 3).to(device)
    
    # Perform few-shot classification
    predictions = FewShotLearning.prototypical_networks(
        support_set, support_labels, query_set, feature_extractor
    )
    
    accuracy = (predictions == query_labels).float().mean().item() * 100
    print(f"Few-shot accuracy: {accuracy:.2f}%")
    
    return predictions

# Example 3: Domain Adaptation
def domain_adaptation_example():
    """Example: Domain adaptation with MMD loss."""
    
    print("\n" + "=" * 50)
    print("Domain Adaptation Example")
    print("=" * 50)
    
    # Create source and target features
    source_features = torch.randn(100, 512).to(device)
    target_features = torch.randn(100, 512).to(device) + 0.5  # Shifted distribution
    
    # Compute MMD before adaptation
    mmd_before = DomainAdaptation.maximum_mean_discrepancy(
        source_features, target_features
    )
    print(f"MMD before adaptation: {mmd_before.item():.4f}")
    
    # Simple adaptation: align means
    source_mean = source_features.mean(0)
    target_mean = target_features.mean(0)
    adapted_target = target_features - target_mean + source_mean
    
    # Compute MMD after adaptation
    mmd_after = DomainAdaptation.maximum_mean_discrepancy(
        source_features, adapted_target
    )
    print(f"MMD after adaptation: {mmd_after.item():.4f}")
    print(f"Improvement: {(mmd_before - mmd_after).item():.4f}")
    
    return mmd_before, mmd_after

# Run examples
if __name__ == "__main__":
    # Run image classification example
    model, history = image_classification_example()
    
    # Run few-shot learning example
    predictions = few_shot_learning_example()
    
    # Run domain adaptation example
    mmd_before, mmd_after = domain_adaptation_example()

## 7. Visualization and Analysis

In [ ]:
def visualize_transfer_learning_results(history: Dict[str, List]) -> None:
    """Visualize transfer learning training history."""
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Loss curves
    axes[0, 0].plot(history['train_loss'], label='Train Loss', alpha=0.8)
    axes[0, 0].plot(history['val_loss'], label='Val Loss', alpha=0.8)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[0, 1].plot(history['train_acc'], label='Train Acc', alpha=0.8)
    axes[0, 1].plot(history['val_acc'], label='Val Acc', alpha=0.8)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Learning rate schedule (if available)
    if 'lr' in history:
        axes[1, 0].plot(history['lr'])
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Learning Rate')
        axes[1, 0].set_title('Learning Rate Schedule')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Phase indicators (if using gradual unfreezing)
    if 'phase' in history:
        phases = history['phase']
        unique_phases = list(set(phases))
        colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_phases)))
        
        for phase, color in zip(unique_phases, colors):
            phase_epochs = [i for i, p in enumerate(phases) if p == phase]
            axes[1, 1].scatter(phase_epochs, 
                             [history['val_acc'][i] for i in phase_epochs],
                             c=[color], label=f'Phase {phase}', s=30, alpha=0.7)
        
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Validation Accuracy')
        axes[1, 1].set_title('Training Phases')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create sample visualization
sample_history = {
    'train_loss': np.random.exponential(0.5, 50)[::-1] + np.random.normal(0, 0.05, 50),
    'val_loss': np.random.exponential(0.5, 50)[::-1] + np.random.normal(0, 0.08, 50),
    'train_acc': 100 * (1 - np.random.exponential(0.3, 50)[::-1]) + np.random.normal(0, 2, 50),
    'val_acc': 100 * (1 - np.random.exponential(0.3, 50)[::-1]) + np.random.normal(0, 3, 50),
    'phase': [1]*10 + [2]*10 + [3]*10 + [4]*10 + [5]*10
}

visualize_transfer_learning_results(sample_history)

## Summary

This comprehensive transfer learning suite provides:

1. **Universal Framework**: Support for multiple modalities (image, text, tabular, multimodal)
2. **Advanced Fine-tuning**: Gradual unfreezing, discriminative learning rates, cyclical strategies
3. **Domain Adaptation**: DANN, MMD, CORAL for distribution alignment
4. **Few-Shot Learning**: Prototypical networks, matching networks, MAML
5. **Model Fusion**: Ensemble techniques, knowledge distillation, stacking
6. **Production Ready**: Configurable, modular, and extensible design

The framework can be easily extended for specific use cases and integrated into production pipelines.